In [1]:
import pandas as pd

def _generate_sliding_windows(start_date, end_date, icl, ocl, stride, prefix, freq='D'):
    """
    Generate sliding window bounds for daily or monthly series.
    - freq: 'D' for daily data, 'M' for monthly data
    """
    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    # Select offset unit based on frequency
    freq_upper = freq.upper()
    if freq_upper in ['D', 'DAILY']:
        step = lambda x: pd.DateOffset(days=x)
    elif freq_upper in ['M', 'MONTHLY']:
        step = lambda x: pd.DateOffset(months=x)
    else:
        raise ValueError("freq must be either 'D' (daily) or 'M' (monthly)")

    records = []
    window_idx = 1
    current_input_start = start

    while True:
        # Calculate window boundary dates using frequency offsets
        current_input_end = current_input_start + step(icl - 1)
        current_output_start = current_input_end + step(1)
        current_output_end = current_output_start + step(ocl - 1)

        # Stop condition: total window exceeds the end date
        if current_output_end > end:
            break

        records.append({
            'WINDOW': f"{prefix}_WINDOW_{window_idx}",
            f'{prefix}_START_INPUT': current_input_start.strftime('%Y-%m-%d'),
            f'{prefix}_END_INPUT': current_input_end.strftime('%Y-%m-%d'),
            'OUTPUT_START': current_output_start.strftime('%Y-%m-%d'),
            'OUTPUT_END': current_output_end.strftime('%Y-%m-%d')
        })

        window_idx += 1
        current_input_start += step(stride)

    return pd.DataFrame(records)


def get_train_windows(train_start_date, train_end_date, icl, ocl, stride, freq='D'):
    return _generate_sliding_windows(
        train_start_date, train_end_date, icl, ocl, stride, prefix='TRAIN', freq=freq
    )


def get_val_windows(val_start_date, val_end_date, icl, ocl, stride, freq='D'):
    return _generate_sliding_windows(
        val_start_date, val_end_date, icl, ocl, stride, prefix='VAL', freq=freq
    )

In [21]:
train_df=get_train_windows('2023-04-01','2026-04-30',365,100,1,freq='D')

In [22]:
val_df=get_val_windows('2025-04-01','2026-07-31',365,100,1,freq='D')

In [24]:
type(train_df)

pandas.core.frame.DataFrame

In [25]:
type(val_df)

pandas.core.frame.DataFrame

In [36]:
train_df["OUTPUT_END"].max()

'2026-04-30'

In [35]:
train_df.loc[train_df["OUTPUT_END"]>='2026-04-01',:].head()

,WINDOW,TRAIN_START_INPUT,TRAIN_END_INPUT,OUTPUT_START,OUTPUT_END
632,TRAIN_WINDOW_633,2024-12-23,2025-12-22,2025-12-23,2026-04-01
633,TRAIN_WINDOW_634,2024-12-24,2025-12-23,2025-12-24,2026-04-02
634,TRAIN_WINDOW_635,2024-12-25,2025-12-24,2025-12-25,2026-04-03
635,TRAIN_WINDOW_636,2024-12-26,2025-12-25,2025-12-26,2026-04-04
636,TRAIN_WINDOW_637,2024-12-27,2025-12-26,2025-12-27,2026-04-05


In [34]:
val_df.head()

,WINDOW,VAL_START_INPUT,VAL_END_INPUT,OUTPUT_START,OUTPUT_END
0,VAL_WINDOW_1,2025-04-01,2026-03-31,2026-04-01,2026-07-09
1,VAL_WINDOW_2,2025-04-02,2026-04-01,2026-04-02,2026-07-10
2,VAL_WINDOW_3,2025-04-03,2026-04-02,2026-04-03,2026-07-11
3,VAL_WINDOW_4,2025-04-04,2026-04-03,2026-04-04,2026-07-12
4,VAL_WINDOW_5,2025-04-05,2026-04-04,2026-04-05,2026-07-13


In [37]:
train_df.to_csv(r"Training_sliding_windows.csv",index=False)
val_df.to_csv(r"Validation_sliding_windows.csv",index=False)

In [38]:
from dateutil.relativedelta import relativedelta

In [39]:
pd.to_datetime('2026-07-31') + relativedelta(days=-465)

Timestamp('2025-04-22 00:00:00')